In [ ]:
"""
Convert PDF into csv

Step 1/5

"""

FileName = 'ACCTREAD'

from docx import Document

from docx.shared import Inches

import os

import tabula

### scrap PDF file pages 30-51 into an csv output file

if FileName == 'ACCTREAD':
    pages_no = '91-93'
elif FileName == 'STMTEMAIL':
    pages_no = '30-51'
    
tabula.convert_into("file:///C:/Users/miw/Desktop/py3/DataProgram/CDE754_File_Formats_Issuing_TRG_01_18_CUS.pdf",\
                         "Cadencie_File_Definitions.csv", output_format ="csv", pages =pages_no, stream = True)





In [ ]:
"""This cell is not working just yet"""

from tabula.io import read_pdf

#df = read_pdf('file.pdf', pages='all')

# rxn_options={'header':None,
#                        'names':['Attribut','Position', 'Length',
#                                              'Edit','Content', 'Description']}

df = read_pdf("file:///C:/Users/miw/Desktop/py3/DataProgram/CDE754_File_Formats_Issuing_TRG_01_18_CUS.pdf",\
                   
                       pages ='91-93')




In [ ]:
import pandas as pd

combined_df = pd.concat(df[1:])

In [ ]:
combined_df.to_csv("temp001.csv", index = None)

In [ ]:
"""
Precursor : Manually tidy up the csv files




Step 2/5

"""

import pandas as pd

### This line enabales us to extract 2 lines of header
#df = pd.read_csv("Cadencie_File_Definitions.csv", encoding ='ISO8859', header=[0,1])

df = pd.read_csv("Cadencie_File_Definitions.csv", encoding ='ISO8859', header=[0])

In [ ]:
"""
INPUT  : Cadencie_File_Definitions.csv
OUTPUT : somefile.txt

First round cleansing 
    remove all the Blank lines
    remove all the rows with Record = "Type"
    remove all the rows with Record = "Record", Attribute ="Attribute", Position = "Position"


Step 3/5

"""

sv_record_type = ''
sv_attribute = ''
sv_position = ''

with open('somefile.txt', "w", encoding="utf-8") as f:
    if FileName == 'ACCTREAD':
        for row in df.itertuples(): 
            if (str(row.Attribute).strip() == "Attribute" and str(row.Position).strip() == "Position") or \
               (pd.isna(row.Attribute) and pd.isna(row.Position)) \
            :
                pass
            else:
                f.write(str(row.Attribute)+"|"+str(row.Position)+"\n")
                #print(row.Attribute, "\t\t ", row.Position)
    elif FileName == 'STMTEMAIL':
        for row in df.itertuples(): 
            if (str(row.Record) == "Type") or \
                (str(row.Record).strip() == "Record" and str(row.Attribute).strip() == "Attribute" and str(row.Position).strip() == "Position") or \
                (pd.isna(row.Record) and pd.isna(row.Attribute) and pd.isna(row.Position)) \
            :
                pass
            else:
                f.write(str(row.Record)+"|"+str(row.Attribute)+"|"+str(row.Position)+"\n")
                #print(row.Record, "\t ", row.Attribute, "\t\t ", row.Position)

f.close()

#print(row['Attribute'])

In [ ]:
"""

INPUT : somefile.txt

Merging the Attribute column 


OUTPUT : finalfile.txt

Step 4/5

"""

df2 = pd.read_csv("somefile.txt", header= None, sep = "|")

sv_record = ''
sv_attribute = ''
sv_position = ''
#sv_row = tuple()


first_flag = True


with open('finalfile.txt', "w", encoding="utf-8") as f:
    if FileName == 'ACCTREAD':
        for row in df2.itertuples(): 
            if first_flag :
                #sv_row = row
                sv_attribute = row[1]
                sv_position  = row[2]
                first_flag   = False
            elif pd.isna(row[2]) :
                sv_attribute = sv_attribute+ " " +str(row[1])  
            else:
                #print(sv_attribute)
                f.write(str(sv_attribute)+"|"+str(sv_position)+"\n")
                sv_attribute = row[1]
                sv_position  = row[2]

        f.write(str(sv_attribute)+"|"+str(sv_position)+"\n")
    elif FileName == 'STMTEMAIL':
        for row in df2.itertuples(): 
            if first_flag :
                #sv_row = row
                sv_record    = row[1]
                sv_attribute = row[2]
                sv_position  = row[3]
                first_flag   = False
            elif (pd.isna(row[1]) and pd.isna(row[3])) :
                sv_attribute = sv_attribute+ " " +str(row[2])  
            else:
                #print(sv_attribute)
                f.write(str(sv_record)+"|"+str(sv_attribute)+"|"+str(sv_position)+"\n")
               
                sv_record    = row[1]
                sv_attribute = row[2]
                sv_position  = row[3]

        f.write(str(sv_record)+"|"+str(sv_attribute)+"|"+str(sv_position)+"\n")
    
f.close


In [ ]:

"""
 INPUT : finalfile.txt
 
 split the column Position into 2 new columns : start, end
 
 OUTPUT : final_output_icd.csv
 
 Step 5/5


"""
import pandas as pd


### Read finalfile.txt into a dataframe, create header row
if FileName == 'STMTEMAIL':
    df3 = pd.read_csv("finalfile.txt", header= None, sep = "|", names=['Record', 'Attribute', 'Position'])
elif FileName == 'ACCTREAD':
    df3 = pd.read_csv("finalfile.txt", header= None, sep = "|", names=['Attribute', 'Position'])
    

def split_position(row):
    #print(row.Position)
    if "" in str(row.Position) :
        #print(" Position {} , start {}".format(row.Position, row.Position.split("")[0]))
        x = row.Position.split("")[0]
        y = row.Position.split("")[-1]
    elif "-" in str(row.Position) :
        #print(" Position {} , start {}".format(row.Position, row.Position.split("-")[0]))
        x = row.Position.split("-")[0]
        y = row.Position.split("-")[-1]
    else:
        x = row.Position
        y = row.Position
    return x, y
    

#### create 2 new columns 'start' and 'end' by applying function split_position
    
    
# the following code works by creating 2 new columns in df3 = start, end
df3[['start', 'end']] = df3.apply(split_position, axis = 1, result_type = "expand")





# df3['start'] = df3['Position'].apply(lambda x: x.split("")[0]  if "" in str(x) else \
#                       x.split("-")[0] if "-" in str(x) else \
#                       x)


# df3['end']   = df3['Position'].apply(lambda x: x.split("")[-1] if "" in str(x) else \
#                       x.split("-")[-1] if "-" in str(x) else \
#                       x)



df3.to_csv("final_output_icd.csv", index = None)
